# AeroPulse Source Likelihood — 03: Labeling Functions

Fifteen independent heuristics that each vote on one source, replacing the
priority chain in which later rules silently overwrote earlier ones.

## Objective

Improvement plan sections 5, 6 and — the central one — section 3.1.

The plan's diagnosis of the old labeller is correct, and the code confirms
it. `weak_source_labels` initialised every row to `mixed_unknown` and then
assigned, in order, `dust` → `industrial` → `regional_transport` →
`traffic` → `biomass_burning`. Each assignment overwrote the last. A row
with fire evidence *and* transport evidence came out as `biomass_burning`
alone, and the pipeline then called this an "independent likelihood"
architecture. It was not one.

This notebook replaces that with independent labeling functions. Each:

- votes on exactly **one** source
- returns **+1 supports / 0 abstains / −1 contradicts**
- carries a **reliability weight** (section 6), so a FIRMS fire detection
  and a night-time-stagnation guess do not count equally
- **abstains** when its required evidence is unavailable, so missing
  evidence never becomes negative evidence (section 45)

Section 4 demonstrates the old behaviour against the new one on the same
rows, rather than asserting the difference.


> **Scientific constraint, repeated in every notebook because it governs how
> the output may be used.** These are *likelihoods derived from heuristics*,
> not measured source attribution. No public ground-truth attribution
> catalogue exists for this dataset. A model that scores well here has
> reproduced its own labeling system — which is not the same as being right.

In [1]:
# ============================================================================
# SETUP — reusable logic lives in source_toolkit.py, not in cells.
# ============================================================================
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import source_toolkit as st

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)

cfg = st.load_config()
SOURCES = st.SOURCES
print("kernel:", sys.executable)

env: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/.env
project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood
sources: ['biomass_burning', 'traffic', 'industrial', 'dust', 'regional_transport']
kernel: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/bin/python


In [2]:
feature_file = cfg.feature_dir / "source_evidence_features.parquet"
if not feature_file.exists():
    raise FileNotFoundError("Run 02_source_evidence_features.ipynb first.")
df = pd.read_parquet(feature_file)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)
print("rows", f"{len(df):,}", "| columns", df.shape[1])

rows 1,705,252 | columns 258


## 1. The labeling function registry

In [3]:
table = st.labeling_function_table()
print(table.to_string(index=False))
print()
print("functions per source:")
print(table.groupby("source").agg(
    functions=("name", "size"), total_reliability=("reliability", "sum")
).round(2).to_string())

                       name             source  reliability                                           requires                                                                                  description
             LF_FIRE_UPWIND    biomass_burning         0.90                         pm25, upwind_fire_frp_50km Upwind fire radiative power high with elevated PM2.5; contradicts when no upwind fire at all
              LF_FIRE_LOCAL    biomass_burning         0.75                          pm25, fire_count_20km_24h                         Any fire detection within 20 km in the last 24 h with elevated PM2.5
             LF_CROP_SEASON    biomass_burning         0.50                          pm25, crop_burning_window                   Inside the kharif/rabi stubble-burning calendar window with elevated PM2.5
               LF_NO2_RATIO            traffic         0.65                                  no2_level_anomaly                            NO2 level above this station's own normal; con

## 2. Fit thresholds on train only

Every quantile the functions use is measured before the training cutoff.
Fitting them on the full table would let the test period set its own
decision boundaries.

In [4]:
fit_end = df["timestamp_utc"].quantile(cfg.train_fraction)
train_slice = df[df["timestamp_utc"] < fit_end]
print("threshold fit window ends:", fit_end, f"({len(train_slice):,} rows)")

thresholds = st.fit_thresholds(train_slice)
for name, value in thresholds.items():
    print(f"  {name:<22} {value:10.3f}")

threshold fit window ends: 2026-03-22 01:30:00+00:00 (1,193,612 rows)
  pm25_high                  58.500
  pm25_low                   26.200
  wind_high                  11.200
  wind_low                    4.800
  rh_low                     44.000
  no2_anomaly_high            1.856
  no2_anomaly_low             0.417
  so2_anomaly_high            1.852
  co_anomaly_high             1.562
  coarse_high                 1.791
  coarse_low                  0.599
  dust_high                  52.000
  upwind_frp_high            18.730
  upwind_pm25_high           53.000
  consistency_high            0.977
  corr_high                   0.400


### Leakage test on the thresholds

Multiply everything after the cutoff by a large factor. The thresholds must
not move.

In [5]:
tampered = df.copy()
future = tampered["timestamp_utc"] >= fit_end
for column in ("pm25", "cams_no2", "cams_so2", "cams_dust", "wind_speed_10m"):
    if column in tampered.columns:
        tampered.loc[future, column] = tampered.loc[future, column] * 15
refit = st.fit_thresholds(tampered[tampered["timestamp_utc"] < fit_end])
identical = all(
    abs(thresholds[k] - refit[k]) < 1e-6
    for k in thresholds if np.isfinite(thresholds[k]) and np.isfinite(refit[k])
)
print("multiplied every post-cutoff value by 15")
print("thresholds unchanged:", identical)
assert identical, "labeling-function thresholds are leaking future data"

multiplied every post-cutoff value by 15
thresholds unchanged: True


## 3. Apply the functions

In [6]:
t0 = time.time()
votes = st.apply_labeling_functions(df, thresholds)
print(f"applied {votes.shape[1]} functions to {len(votes):,} rows in {time.time() - t0:.1f}s")

summary = pd.DataFrame({
    "supports_pct": (votes == 1).mean() * 100,
    "contradicts_pct": (votes == -1).mean() * 100,
    "abstains_pct": (votes == 0).mean() * 100,
}).round(2)
summary["source"] = [lf.source for lf in st.LABELING_FUNCTIONS]
summary["reliability"] = [lf.reliability for lf in st.LABELING_FUNCTIONS]
print()
print(summary[["source", "reliability", "supports_pct", "contradicts_pct", "abstains_pct"]].to_string())

silent = summary[(summary["supports_pct"] == 0) & (summary["contradicts_pct"] == 0)]
print("\nfunctions that never fired:", list(silent.index) or "none")

applied 15 functions to 1,705,252 rows in 0.5s

                                         source  reliability  supports_pct  contradicts_pct  abstains_pct
LF_FIRE_UPWIND                  biomass_burning         0.90          2.40             8.54         89.06
LF_FIRE_LOCAL                   biomass_burning         0.75          5.98             0.00         94.02
LF_CROP_SEASON                  biomass_burning         0.50          5.74             0.00         94.26
LF_NO2_RATIO                            traffic         0.65         17.22            22.40         60.37
LF_TRAFFIC_DIURNAL                      traffic         0.60          3.33             1.30         95.37
LF_NO2_PM25_CORR                        traffic         0.55         36.57            22.45         40.98
LF_DUST_CAMS                               dust         0.80         18.08             0.00         81.92
LF_DUST_METEOROLOGY                        dust         0.60          0.54             0.00         99.4

### Do the functions within a source agree?

Correlated labeling functions inflate apparent consensus — three functions
that always agree carry the information of one. Disagreement is not a
failure either; it is what the conflict score is for.

In [7]:
for source in SOURCES:
    names = [lf.name for lf in st.LABELING_FUNCTIONS if lf.source == source]
    if len(names) < 2:
        continue
    corr = votes[names].corr()
    print(f"\n{source}: pairwise vote correlation")
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            value = corr.loc[a, b]
            flag = "  <- highly correlated" if abs(value) > 0.6 else ""
            print(f"    {a:<30} x {b:<30} {value:+.3f}{flag}")


biomass_burning: pairwise vote correlation
    LF_FIRE_UPWIND                 x LF_FIRE_LOCAL                  +0.140
    LF_FIRE_UPWIND                 x LF_CROP_SEASON                 -0.261
    LF_FIRE_LOCAL                  x LF_CROP_SEASON                 +0.316

traffic: pairwise vote correlation
    LF_NO2_RATIO                   x LF_TRAFFIC_DIURNAL             +0.053
    LF_NO2_RATIO                   x LF_NO2_PM25_CORR               +0.056
    LF_TRAFFIC_DIURNAL             x LF_NO2_PM25_CORR               +0.037

industrial: pairwise vote correlation
    LF_SO2_INDUSTRIAL              x LF_INDUSTRIAL_NOCTURNAL        +0.020
    LF_SO2_INDUSTRIAL              x LF_INDUSTRIAL_WEEKEND          +0.048
    LF_INDUSTRIAL_NOCTURNAL        x LF_INDUSTRIAL_WEEKEND          +0.022

dust: pairwise vote correlation
    LF_DUST_CAMS                   x LF_DUST_METEOROLOGY            +0.050
    LF_DUST_CAMS                   x LF_COARSE_RATIO                +0.435
    LF_DUST_METEOROLOGY

## 4. Old labeller versus new — the same rows, side by side

This is the architectural claim, tested. The old rule is reproduced exactly
as it was written, including the overwrite order.

In [8]:
def legacy_weak_source_labels(frame, thr):
    """The previous implementation, reproduced verbatim for comparison."""
    pm = pd.to_numeric(frame["pm25"], errors="coerce")
    high_pm = pm >= thr["pm25_high"]
    windy = pd.to_numeric(frame["wind_speed_10m"], errors="coerce") >= thr["wind_high"]
    dry = pd.to_numeric(frame["relative_humidity_2m"], errors="coerce") <= thr["rh_low"]
    fire = frame["fire_active_flag"].fillna(0).astype(bool)
    hour = frame["hour"]
    weekend = frame["is_weekend"].astype(bool)
    season = frame["season"]
    stagnant = frame["stagnation_flag"].astype(bool)
    traffic = hour.isin(range(6, 11)) & (~weekend) & high_pm & (~fire)

    labels = pd.Series("mixed_unknown", index=frame.index, dtype=object)
    labels[windy & dry & high_pm & (season == "pre_monsoon")] = "dust"
    labels[high_pm & stagnant & (~fire) & ((hour.isin(list(range(22, 24)) + list(range(0, 6)))) | weekend)] = "industrial"
    labels[windy & high_pm] = "regional_transport"
    labels[traffic.fillna(False)] = "traffic"
    labels[fire & high_pm] = "biomass_burning"
    return labels

legacy = legacy_weak_source_labels(df, thresholds)
print("legacy single-label distribution (%):")
counts = (legacy.value_counts() / len(legacy) * 100).round(2)
for cls in ["mixed_unknown", "biomass_burning", "traffic", "regional_transport",
            "industrial", "dust"]:
    print(f"  {cls:<20} {counts.get(cls, 0.0):6.2f}")
print("\nrows the legacy rule could assign to more than one source: 0 (by construction)")

missing = [c for c in ["dust", "industrial", "traffic", "regional_transport"]
           if counts.get(c, 0.0) < 0.5]
if missing:
    print(f"\nClasses the overwrite chain nearly or entirely erased: {missing}")
    print("They are not rare in the evidence — they are later in the priority order.")

legacy single-label distribution (%):
  mixed_unknown         85.60
  biomass_burning       13.75
  traffic                0.32
  regional_transport     0.25
  industrial             0.08
  dust                   0.00

rows the legacy rule could assign to more than one source: 0 (by construction)

Classes the overwrite chain nearly or entirely erased: ['dust', 'industrial', 'traffic', 'regional_transport']
They are not rare in the evidence — they are later in the priority order.


In [9]:
# How much evidence did the overwrite chain discard? Count rows where the
# legacy rule picked one source while other labeling functions were also
# voting +1 for a different source.
supports = pd.DataFrame({
    source: (votes[[lf.name for lf in st.LABELING_FUNCTIONS if lf.source == source]] == 1).any(axis=1)
    for source in SOURCES
})
n_supported = supports.sum(axis=1)

assigned = legacy[legacy != "mixed_unknown"]
overwritten = 0
for source in SOURCES:
    rows = legacy == source
    others = supports.loc[rows, [s for s in SOURCES if s != source]].any(axis=1)
    overwritten += int(others.sum())

print(f"rows the legacy rule assigned to a source : {len(assigned):,}")
print(f"of those, rows where another source also had supporting evidence: {overwritten:,} "
      f"({overwritten / max(len(assigned), 1):.1%})")
print(f"\nrows with 2+ sources supported by the new functions: "
      f"{int((n_supported > 1).sum()):,} ({float((n_supported > 1).mean()):.1%})")
print("distribution of simultaneously supported sources:")
print(n_supported.value_counts().sort_index().to_string())

rows the legacy rule assigned to a source : 245,508
of those, rows where another source also had supporting evidence: 217,962 (88.8%)

rows with 2+ sources supported by the new functions: 511,112 (30.0%)
distribution of simultaneously supported sources:
0    504658
1    689482
2    393097
3    104574
4     12920
5       521


## 5. Save the votes

In [10]:
out = cfg.label_dir / "labeling_function_votes.parquet"
votes.to_parquet(out, index=False)
table.to_csv(cfg.label_dir / "labeling_functions.csv", index=False)

manifest = {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "label_schema_version": st.SCHEMA["label_schema_version"],
    "n_functions": len(st.LABELING_FUNCTIONS),
    "sources": list(SOURCES),
    "thresholds": thresholds,
    "threshold_fit_end": str(fit_end),
    "lf_input_columns": list(st.LF_INPUT_COLUMNS),
    "vote_summary": summary.to_dict(orient="index"),
    "legacy_comparison": {
        "legacy_assigned_rows": int(len(assigned)),
        "legacy_rows_with_competing_evidence": int(overwritten),
        "new_rows_multi_source": int((n_supported > 1).sum()),
    },
}
(cfg.label_dir / "labeling_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
print("03 COMPLETE ->", out)
print("Next: 04_source_weak_supervision.ipynb")

03 COMPLETE -> /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/data/source/processed/labels/labeling_function_votes.parquet
Next: 04_source_weak_supervision.ipynb


## Findings

**The plan's central diagnosis is confirmed, and the cost is larger than it
looked.** The legacy priority chain assigned a source to 245,508 rows. On
**217,962 of them — 88.8% — at least one other source also had supporting
evidence** that the overwrite silently discarded.

The legacy class distribution shows how the damage lands:

| legacy class | share |
|---|---|
| mixed_unknown | 85.60% |
| biomass_burning | 13.75% |
| traffic | 0.32% |
| regional_transport | 0.25% |
| industrial | 0.08% |
| **dust** | **0.00%** |

`dust` is not rare in the evidence — `LF_DUST_CAMS` alone fires on 18.1% of
rows. It is simply **first in the overwrite order**, so every dust row that
also looked industrial, windy or fiery was relabelled. The old pipeline was
structurally incapable of producing a dust label, and no amount of model
tuning downstream could have recovered it. That is the concrete cost of the
architecture the plan flagged.

With independent functions, 30.0% of rows have two or more sources
supported, and 118,015 rows have three or more.

**All 15 functions fire, and abstention rates are high by design.**

| function | source | supports | contradicts | abstains |
|---|---|---|---|---|
| LF_FIRE_UPWIND | biomass | 2.40% | 8.54% | 89.06% |
| LF_NO2_PM25_CORR | traffic | 36.57% | 22.45% | 40.98% |
| LF_COARSE_RATIO | dust | 28.59% | 22.72% | 48.70% |
| LF_DUST_CAMS | dust | 18.08% | 0.00% | 81.92% |
| LF_SO2_INDUSTRIAL | industrial | 12.02% | 0.00% | 87.98% |
| LF_TRANSPORT_UPWIND | regional | 0.41% | 2.48% | 97.11% |
| LF_INDUSTRIAL_NOCTURNAL | industrial | 0.06% | 0.00% | 99.94% |

Two are close to silent and should be treated as decorative until better
evidence exists: `LF_INDUSTRIAL_NOCTURNAL` fires on 0.06% of rows (the
conjunction of night, stagnation, high PM2.5 and no fire is simply rare),
and `LF_DUST_METEOROLOGY` on 0.54%. `LF_TRANSPORT_UPWIND` supports only
0.41% while contradicting 2.48% — it spends most of its time arguing
*against* transport, which is a legitimate use of the −1 vote but means
regional transport has almost no positive evidence in this dataset.

**The functions within a source are close to independent**, which is what
makes weighted voting meaningful. Highest within-source correlation is
`LF_DUST_CAMS` × `LF_COARSE_RATIO` at +0.435 — two different instruments
pointing at the same physics, mildly correlated, as expected. Everything
else is below 0.32, and `LF_FIRE_UPWIND` × `LF_CROP_SEASON` is *negative*
(−0.261): upwind fire detections and the calendar window disagree more than
they agree, which is worth knowing before trusting either alone.

**Thresholds are leakage-controlled.** Multiplying every post-cutoff PM2.5,
NO2, SO2, dust and wind value by 15 leaves all 16 thresholds bit-identical.